# Hikmat PSX — Public API test

Tests the live backend at **http://47.84.234.2** end-to-end: auth → create chat → stream a reply → read history → read/forget preferences.

**Fill in your `EMAIL` / `PASSWORD` in the config cell**, then run the cells top-to-bottom.

> ⚠️ Don't commit this notebook with real credentials saved in it.

In [ ]:
import json
import requests

# ---- config: EDIT THESE ----
BASE_URL  = "http://47.84.234.2"   # public app URL (nginx proxies the API)
EMAIL     = "you@example.com"       # <-- your email
PASSWORD  = "your-password"         # <-- your password
FULL_NAME = "Test User"             # used only if the account is created

TOKEN = None
USER = None

def auth_headers():
    return {"Authorization": f"Bearer {TOKEN}"}

# quick connectivity check
print(requests.get(f"{BASE_URL}/health", timeout=15).json())

## 1. Authenticate (login, or sign up if new)

In [ ]:
def authenticate():
    global TOKEN, USER
    r = requests.post(f"{BASE_URL}/auth/login",
                      json={"email": EMAIL, "password": PASSWORD}, timeout=20)
    if r.status_code == 200:
        data = r.json(); print("Logged in as", data["email"])
    else:
        print("Login failed (", r.status_code, r.json().get("detail"), ") — signing up...")
        r = requests.post(f"{BASE_URL}/auth/signup",
                          json={"full_name": FULL_NAME, "email": EMAIL, "password": PASSWORD},
                          timeout=20)
        r.raise_for_status()
        data = r.json(); print("Signed up as", data["email"])
    TOKEN = data["token"]; USER = data
    return data

authenticate()
print("user_id:", USER["user_id"])

In [ ]:
# whoami
print(requests.get(f"{BASE_URL}/auth/me", headers=auth_headers(), timeout=20).json())

## 2. Create a chat and stream a reply (SSE)

In [ ]:
chat = requests.post(f"{BASE_URL}/chats", headers=auth_headers(), timeout=20).json()
chat_id = chat["chat_id"]
print("chat_id:", chat_id)

In [ ]:
def send_message(chat_id, message):
    """POST a message and print the streamed answer as it arrives."""
    url = f"{BASE_URL}/chats/{chat_id}/message"
    full = ""
    with requests.post(url, json={"message": message}, headers=auth_headers(),
                       stream=True, timeout=120) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            evt = json.loads(line[5:].strip())
            t = evt.get("type")
            if t == "text":
                print(evt["content"], end="", flush=True)
                full += evt["content"]
            elif t == "preferences":
                print("\n\n[preferences]", evt["content"])
            elif t == "error":
                print("\n[ERROR]", evt["content"])
            elif t == "end":
                print("\n[done]")
    return full

_ = send_message(chat_id, "Compare HBL vs MCB and remember I prefer long-term investing.")

## 3. Read the chat history

In [ ]:
msgs = requests.get(f"{BASE_URL}/chats/{chat_id}/messages", headers=auth_headers(), timeout=20).json()
for m in msgs:
    print(f"[{m['qid']}] Q: {m['question']}")
    print(f"    A: {m['answer'][:200]}...\n")

# all chats (sidebar)
print("chats:", requests.get(f"{BASE_URL}/chats", headers=auth_headers(), timeout=20).json())

## 4. Long-term memory: view + forget preferences

In [ ]:
prefs = requests.get(f"{BASE_URL}/me/preferences", headers=auth_headers(), timeout=20).json()
print(json.dumps(prefs, indent=2))

In [ ]:
# forget one preference (uncomment and set the key you want to remove)
# key = "investment_horizon"
# print(requests.delete(f"{BASE_URL}/me/preferences/{key}", headers=auth_headers(), timeout=20).json())